
# Bengali NLP Safety V26
## Noise-Aware Dual-Script Mixture of Experts

**Protected public-LB champion: V14 = 0.586**

This notebook deliberately breaks from V20–V25. It does **not** tune against the official 500-row
validation set. All model/ensemble/threshold choices are learned from **train-only OOF predictions**.
The official validation set is used once, at the end, as a genuinely untouched check.

### Architecture

1. **BanglaBERT** — native Bengali expert.
2. **BanglishBERT** — Bengali + English / mixed-script expert.
3. **HateBERT** — Romanized-only expert.
4. **Exact-duplicate soft labels** — repeated identical texts are collapsed; conflicting labels become
   probabilities instead of forcing a deterministic model to memorize incompatible targets.
5. **Label-confidence weighting** — ambiguous duplicate labels receive less loss weight.
6. **Script tags** — `[BN]`, `[MIX]`, `[ROM]`.
7. **Dataset-specific UNK tokens** for BanglaBERT/BanglishBERT, selected *inside each fold only*.
8. **Competition-shaped loss** — 50% toxicity objective + 50% six-attribute objective.
9. **Soft hierarchy penalty** because the six attributes are observed only inside toxic examples in the
   supplied labeled data.
10. **Script-gated OOF ensemble** with thresholds selected only from train OOF.

### Important

- No external labeled toxicity dataset is used.
- Validation is not used for tuning.
- V14 is never blended into V26; it is included only for final comparison.
- This is substantially more expensive than earlier notebooks: 15 fold-models total.
- On a Colab T4, expect roughly **1–2+ hours**, depending on runtime speed.

Run all cells and send `v26_results.zip` back to ChatGPT **before submitting**.


In [ ]:

# 1. Upload V26 data
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v26_data.zip" in uploaded, "Upload the supplied v26_data.zip."

DATA_DIR = pathlib.Path("/content/v26_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v26_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install dependencies + official Bangla normalization
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn",
    "pandas",
    "numpy",
    "tqdm",
    "safetensors",
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/csebuetnlp/normalizer"
])

from transformers import AutoConfig, AutoTokenizer
from normalizer import normalize

MODELS = {
    "bangla": "csebuetnlp/banglabert",
    "banglish": "csebuetnlp/banglishbert",
    "hate": "GroNLP/hateBERT",
}

for name, model_name in MODELS.items():
    print("Preflight:", name, model_name)
    _ = AutoConfig.from_pretrained(model_name)
    _ = AutoTokenizer.from_pretrained(model_name)

print("Normalizer:", normalize("আমি   ভালো আছি।"))
print("✅ All three experts available.")


In [ ]:

# 3. Imports, settings, data
import gc, re, math, random, json, zipfile, unicodedata, collections
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)
from normalizer import normalize

SEED = 42
N_FOLDS = 5
MAX_LENGTH = 128
EPOCHS = 3
SNAPSHOT_EPOCHS = [2, 3]

TRAIN_BATCH_SIZE = 8
ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 32

LR_BACKBONE = 1.5e-5
LR_HEAD = 3.0e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
HIERARCHY_LAMBDA = 0.05

MAX_NEW_UNK_TOKENS = 64
MIN_UNK_FREQ = 2
SCRIPT_TOKENS = ["[BN]", "[MIX]", "[ROM]"]

LABELS = [
    "is_toxic",
    "implicit_toxicity",
    "sarcasm",
    "contains_profanity",
    "threatening",
    "identity_attack",
    "sexual_content",
]

MODELS = {
    "bangla": "csebuetnlp/banglabert",
    "banglish": "csebuetnlp/banglishbert",
    "hate": "GroNLP/hateBERT",
}

RUN_DIR = Path("/content/v26_output")
RUN_DIR.mkdir(exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
assert torch.cuda.is_available(), "Enable a T4 GPU."

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
sample = pd.read_csv(DATA_DIR/"sample_submission.csv", dtype={"ID":str})
champion = pd.read_csv(DATA_DIR/"submission_v14_champion.csv", dtype={"ID":str})

assert len(train) > 3000 and len(valid)==500 and len(test)==1000
assert test["ID"].tolist() == champion["ID"].tolist()

print("train / validation / test:", len(train), len(valid), len(test))
print(train[LABELS].mean().round(4))


In [ ]:

# 4. Text normalization, script audit, duplicate-label audit
BN_RE = re.compile(r"[\u0980-\u09FF]")
LAT_RE = re.compile(r"[A-Za-z]")

def basic_clean(x):
    x = unicodedata.normalize("NFKC", str(x))
    return re.sub(r"\s+", " ", x).strip()

def normalized_key(x):
    return basic_clean(x).lower()

def script_type(x):
    x = str(x)
    b = bool(BN_RE.search(x))
    l = bool(LAT_RE.search(x))
    if b and l:
        return "mixed"
    if b:
        return "native"
    return "roman"

def script_tag(s):
    return {"native":"[BN]", "mixed":"[MIX]", "roman":"[ROM]"}[s]

def official_norm(x):
    x = basic_clean(x)
    try:
        x = normalize(x)
    except Exception:
        pass
    return re.sub(r"\s+", " ", x).strip()

for df in [train, valid, test]:
    df["norm_key"] = df["text"].map(normalized_key)
    df["script"] = df["text"].map(script_type)
    df["bn_text"] = [
        f"{script_tag(s)} {official_norm(t)}"
        for s,t in zip(df["script"], df["text"])
    ]
    df["hate_text"] = [
        f"{script_tag(s)} {basic_clean(t)}"
        for s,t in zip(df["script"], df["text"])
    ]

print("Script distribution")
audit_script = pd.DataFrame({
    "train": train["script"].value_counts(normalize=True),
    "validation": valid["script"].value_counts(normalize=True),
    "test": test["script"].value_counts(normalize=True),
}).fillna(0)
display(audit_script.round(4))

grp = train.groupby("norm_key", sort=False)
dup_sizes = grp.size()
dup_groups = int((dup_sizes > 1).sum())
extra_duplicate_rows = int((dup_sizes - 1).clip(lower=0).sum())

conflicts = {}
for label in LABELS:
    nunique = grp[label].nunique()
    conflicts[label] = int((nunique > 1).sum())

audit = {
    "train_rows": int(len(train)),
    "unique_normalized_texts": int(train["norm_key"].nunique()),
    "duplicate_groups": dup_groups,
    "extra_duplicate_rows": extra_duplicate_rows,
    "conflicting_duplicate_groups_by_label": conflicts,
}
print(json.dumps(audit, indent=2))
(RUN_DIR/"v26_data_audit.json").write_text(json.dumps(audit, indent=2))


In [ ]:

# 5. Collapse exact duplicates into soft-label training examples
records = []
key_to_gid = {}

for gid, (key, g) in enumerate(train.groupby("norm_key", sort=False)):
    first = g.iloc[0]
    targets = g[LABELS].mean().to_numpy(np.float32)

    confidence = np.maximum(
        0.25,
        2.0 * np.abs(targets - 0.5)
    ).astype(np.float32)

    rec = {
        "group_id": gid,
        "norm_key": key,
        "text": first["text"],
        "script": first["script"],
        "bn_text": first["bn_text"],
        "hate_text": first["hate_text"],
        "group_size": int(len(g)),
    }
    for j,l in enumerate(LABELS):
        rec[l] = float(targets[j])
        rec[f"conf_{l}"] = float(confidence[j])

    records.append(rec)
    key_to_gid[key] = gid

unique_train = pd.DataFrame(records)

unique_train["sample_weight"] = np.sqrt(
    unique_train["group_size"].astype(float)
)
unique_train["sample_weight"] /= unique_train["sample_weight"].mean()

train["group_id"] = train["norm_key"].map(key_to_gid).astype(int)

print("Original train:", len(train))
print("Unique soft-label train:", len(unique_train))
display(unique_train["group_size"].describe())

weighted_means = {}
for l in LABELS:
    weighted_means[l] = np.average(
        unique_train[l],
        weights=unique_train["group_size"]
    )
display(pd.Series(weighted_means).round(4))


In [ ]:

# 6. Train-only balanced 5-fold split over UNIQUE texts
effective_label_mass = np.column_stack([
    unique_train[l].to_numpy(float)
    * unique_train["group_size"].to_numpy(float)
    for l in LABELS
])
effective_rows = unique_train["group_size"].to_numpy(float)

global_mass = effective_label_mass.sum(axis=0)
total_rows = effective_rows.sum()
rarity = 1.0 / np.sqrt(np.maximum(global_mass, 1.0))

priority = (
    (effective_label_mass * rarity).sum(axis=1)
    + 0.005 * effective_rows
)

rng = np.random.default_rng(SEED)
order = np.argsort(
    -(priority + rng.random(len(unique_train))*1e-8)
)

fold_mass = np.zeros((N_FOLDS, len(LABELS)), dtype=float)
fold_rows = np.zeros(N_FOLDS, dtype=float)
fold_unique = np.zeros(N_FOLDS, dtype=float)
fold_assignment = np.full(len(unique_train), -1, dtype=int)

for idx in order:
    vec = effective_label_mass[idx]
    nrows = effective_rows[idx]

    costs = []
    for f in range(N_FOLDS):
        m = fold_mass.copy()
        r = fold_rows.copy()
        u = fold_unique.copy()

        m[f] += vec
        r[f] += nrows
        u[f] += 1

        label_share = m / np.maximum(global_mass[None,:], 1.0)
        row_share = r / max(total_rows, 1.0)
        unique_share = u / len(unique_train)

        cost = (
            np.mean((label_share - 1/N_FOLDS)**2)
            + 0.25*np.mean((row_share - 1/N_FOLDS)**2)
            + 0.10*np.mean((unique_share - 1/N_FOLDS)**2)
        )
        costs.append(cost)

    best = int(np.argmin(costs))
    fold_assignment[idx] = best
    fold_mass[best] += vec
    fold_rows[best] += nrows
    fold_unique[best] += 1

unique_train["fold"] = fold_assignment

gid_to_fold = unique_train.set_index("group_id")["fold"]
train["fold"] = train["group_id"].map(gid_to_fold).astype(int)

assert set(unique_train["fold"]) == set(range(N_FOLDS))
assert train.groupby("norm_key")["fold"].nunique().max() == 1

print("Unique counts:", unique_train.groupby("fold").size().to_dict())
print("Original-row mass:", train.groupby("fold").size().to_dict())
display(train.groupby("fold")[LABELS].sum())


In [ ]:

# 7. Fold-local dataset-specific UNK token discovery
TOKEN_RE = re.compile(r"\S+")

def select_unk_tokens(
    model_name,
    texts,
    max_new=MAX_NEW_UNK_TOKENS,
    min_freq=MIN_UNK_FREQ
):
    # Uses ONLY the current fold's training texts.
    tok = AutoTokenizer.from_pretrained(model_name)
    counts = collections.Counter()

    for text in texts:
        for word in TOKEN_RE.findall(str(text)):
            word = word.strip()
            if len(word) < 2 or len(word) > 32:
                continue
            if word.startswith("http") or word.startswith("www."):
                continue
            counts[word] += 1

    candidates = []
    unk_id = tok.unk_token_id

    for word, freq in counts.most_common():
        if freq < min_freq:
            break
        ids = tok.encode(word, add_special_tokens=False)
        if unk_id is not None and unk_id in ids:
            candidates.append(word)
            if len(candidates) >= max_new:
                break

    return candidates

probe_rows = unique_train[unique_train.fold != 0]
for expert in ["bangla", "banglish"]:
    toks = select_unk_tokens(
        MODELS[expert],
        probe_rows["bn_text"].tolist(),
        max_new=20
    )
    print(expert, "sample fold-local UNK additions:", toks[:20])


In [ ]:

# 8. Model and loss
METRIC_WEIGHTS = torch.tensor(
    [0.5] + [0.5/6.0]*6,
    dtype=torch.float32
)

class SoftLabelDataset(Dataset):
    def __init__(self, frame, tokenizer, text_col, with_labels=True):
        self.frame = (
            frame.reset_index(drop=False)
            .rename(columns={"index":"source_index"})
        )
        self.texts = self.frame[text_col].astype(str).tolist()
        self.tokenizer = tokenizer
        self.with_labels = with_labels

        if with_labels:
            self.targets = self.frame[LABELS].to_numpy(np.float32)
            self.conf = self.frame[
                [f"conf_{l}" for l in LABELS]
            ].to_numpy(np.float32)
            self.sw = self.frame["sample_weight"].to_numpy(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        enc["source_index"] = int(
            self.frame.iloc[i]["source_index"]
        )

        if self.with_labels:
            enc["labels"] = self.targets[i]
            enc["confidence"] = self.conf[i]
            enc["sample_weight"] = self.sw[i]

        return enc

class ExternalDataset(Dataset):
    def __init__(self, frame, tokenizer, text_col):
        self.frame = (
            frame.reset_index(drop=False)
            .rename(columns={"index":"source_index"})
        )
        self.texts = self.frame[text_col].astype(str).tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        enc["source_index"] = int(
            self.frame.iloc[i]["source_index"]
        )
        return enc

class SmartCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, items):
        labels = confidence = sample_weight = None

        source_index = torch.tensor(
            [x.pop("source_index") for x in items],
            dtype=torch.long
        )

        if "labels" in items[0]:
            labels = torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )
            confidence = torch.tensor(
                np.stack([x.pop("confidence") for x in items]),
                dtype=torch.float32
            )
            sample_weight = torch.tensor(
                [x.pop("sample_weight") for x in items],
                dtype=torch.float32
            )

        batch = self.tokenizer.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )
        batch["source_index"] = source_index

        if labels is not None:
            batch["labels"] = labels
            batch["confidence"] = confidence
            batch["sample_weight"] = sample_weight

        return batch

class NoiseAwareClassifier(nn.Module):
    def __init__(self, model_name, vocab_size):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_name)

        old_n = self.base.get_input_embeddings().num_embeddings
        if vocab_size != old_n:
            self.base.resize_token_embeddings(vocab_size)

        hidden = self.base.config.hidden_size
        self.dropouts = nn.ModuleList([
            nn.Dropout(0.10),
            nn.Dropout(0.20),
            nn.Dropout(0.30),
        ])
        self.classifier = nn.Linear(hidden*2, len(LABELS))

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None
    ):
        kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        out = self.base(**kwargs)
        h = out.last_hidden_state

        cls = h[:,0]
        mask = attention_mask.unsqueeze(-1).float()
        mean = (
            (h*mask).sum(1)
            / mask.sum(1).clamp_min(1.0)
        )

        feat = torch.cat([cls, mean], dim=-1)
        logits = torch.stack(
            [self.classifier(d(feat)) for d in self.dropouts],
            dim=0
        ).mean(0)
        return logits

def compute_pos_weight(frame):
    sizes = frame["group_size"].to_numpy(float)
    y = frame[LABELS].to_numpy(float)

    pos = (y * sizes[:,None]).sum(axis=0)
    neg = ((1-y) * sizes[:,None]).sum(axis=0)

    pw = np.sqrt(neg / np.maximum(pos, 1.0))
    return np.clip(pw, 0.75, 4.0).astype(np.float32)

def competition_shaped_loss(
    logits,
    y,
    confidence,
    sample_weight,
    pos_weight
):
    raw = torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(),
        y,
        pos_weight=pos_weight,
        reduction="none",
    )

    raw = raw * confidence
    mw = METRIC_WEIGHTS.to(logits.device).unsqueeze(0)
    base_loss = (raw * mw).sum(dim=1)

    p = torch.sigmoid(logits.float())
    hierarchy = torch.relu(
        p[:,1:] - p[:,:1]
    ).pow(2).mean(dim=1)

    per_sample = (
        base_loss
        + HIERARCHY_LAMBDA * hierarchy
    )

    sw = (
        sample_weight
        / sample_weight.mean().clamp_min(1e-6)
    )
    return (per_sample * sw).mean()

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    probs = []
    indices = []

    for batch in loader:
        idx = batch.pop("source_index")
        batch.pop("labels", None)
        batch.pop("confidence", None)
        batch.pop("sample_weight", None)

        inputs = {
            k:v.to("cuda", non_blocking=True)
            for k,v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(**inputs)

        probs.append(
            torch.sigmoid(logits.float()).cpu().numpy()
        )
        indices.append(idx.numpy())

    return np.concatenate(indices), np.concatenate(probs)


In [ ]:

# 9. Train one expert across 5 folds
fold_seeds = [42, 3407, 2026, 17, 1337]

def build_tokenizer(expert, fold_train):
    model_name = MODELS[expert]
    tok = AutoTokenizer.from_pretrained(model_name)

    tok.add_special_tokens({
        "additional_special_tokens": SCRIPT_TOKENS
    })

    added_unk = []
    if expert in ["bangla", "banglish"]:
        added_unk = select_unk_tokens(
            model_name,
            fold_train["bn_text"].tolist(),
            max_new=MAX_NEW_UNK_TOKENS,
            min_freq=MIN_UNK_FREQ,
        )
        if added_unk:
            tok.add_tokens(added_unk)

    return tok, added_unk

def train_expert(expert):
    model_name = MODELS[expert]
    text_col = "hate_text" if expert=="hate" else "bn_text"

    oof = np.full(
        (len(unique_train), len(LABELS)),
        np.nan,
        dtype=np.float32
    )
    valid_fold_preds = []
    test_fold_preds = []
    histories = []
    vocab_reports = []

    for fold in range(N_FOLDS):
        if expert == "hate":
            tr = unique_train[
                (unique_train.fold != fold)
                & (unique_train.script == "roman")
            ].copy()
            va = unique_train[
                (unique_train.fold == fold)
                & (unique_train.script == "roman")
            ].copy()

            valid_ext = valid[valid.script=="roman"].copy()
            test_ext = test[test.script=="roman"].copy()
        else:
            tr = unique_train[
                unique_train.fold != fold
            ].copy()
            va = unique_train[
                unique_train.fold == fold
            ].copy()

            valid_ext = valid.copy()
            test_ext = test.copy()

        assert len(tr)>0 and len(va)>0

        cache = RUN_DIR/f"{expert}_fold_{fold}.npz"
        hist_path = RUN_DIR/f"{expert}_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache, allow_pickle=False) as z:
                va_idx = z["va_idx"]
                va_prob = z["va_prob"]
                valid_idx = z["valid_idx"]
                valid_prob = z["valid_prob"]
                test_idx = z["test_idx"]
                test_prob = z["test_prob"]

            oof[va_idx] = va_prob

            valid_full = np.full(
                (len(valid),len(LABELS)),
                np.nan,dtype=np.float32
            )
            valid_full[valid_idx] = valid_prob

            test_full = np.full(
                (len(test),len(LABELS)),
                np.nan,dtype=np.float32
            )
            test_full[test_idx] = test_prob

            valid_fold_preds.append(valid_full)
            test_fold_preds.append(test_full)

            if hist_path.exists():
                histories.extend(
                    pd.read_csv(hist_path).to_dict("records")
                )

            print(f"✅ reuse {expert} fold {fold}")
            continue

        print("\n"+"="*76)
        print(
            f"{expert.upper()} fold {fold} | "
            f"train unique={len(tr)} | oof unique={len(va)}"
        )

        seed_everything(fold_seeds[fold])
        gc.collect()
        torch.cuda.empty_cache()

        tokenizer, added_unk = build_tokenizer(expert, tr)
        vocab_reports.append({
            "expert":expert,
            "fold":fold,
            "added_unk_count":len(added_unk),
            "added_unk_tokens":" | ".join(added_unk),
        })

        collator = SmartCollator(tokenizer)

        tl = DataLoader(
            SoftLabelDataset(
                tr,tokenizer,text_col,True
            ),
            batch_size=TRAIN_BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator,
        )
        vl = DataLoader(
            SoftLabelDataset(
                va,tokenizer,text_col,True
            ),
            batch_size=EVAL_BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator,
        )
        val_ext_loader = DataLoader(
            ExternalDataset(
                valid_ext,tokenizer,text_col
            ),
            batch_size=EVAL_BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator,
        )
        test_ext_loader = DataLoader(
            ExternalDataset(
                test_ext,tokenizer,text_col
            ),
            batch_size=EVAL_BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=collator,
        )

        model = NoiseAwareClassifier(
            model_name,
            len(tokenizer)
        ).to("cuda")

        pos_weight = torch.tensor(
            compute_pos_weight(tr),
            dtype=torch.float32,
            device="cuda",
        )
        print(
            "pos_weight:",
            dict(
                zip(
                    LABELS,
                    np.round(
                        pos_weight.cpu().numpy(),3
                    )
                )
            )
        )
        print("fold-local new tokens:", len(added_unk))

        head_params = []
        backbone_params = []

        for n,p in model.named_parameters():
            if not p.requires_grad:
                continue
            if n.startswith("classifier") or n.startswith("dropouts"):
                head_params.append(p)
            else:
                backbone_params.append(p)

        optimizer = torch.optim.AdamW([
            {
                "params":backbone_params,
                "lr":LR_BACKBONE,
                "weight_decay":WEIGHT_DECAY
            },
            {
                "params":head_params,
                "lr":LR_HEAD,
                "weight_decay":WEIGHT_DECAY
            },
        ])

        total_steps = (
            math.ceil(len(tl)/ACCUM_STEPS)
            * EPOCHS
        )
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            int(total_steps*WARMUP_RATIO),
            total_steps
        )
        scaler = torch.amp.GradScaler("cuda")

        snapshot_oof = []
        snapshot_valid = []
        snapshot_test = []
        fold_hist = []

        for epoch in range(1,EPOCHS+1):
            model.train()
            optimizer.zero_grad(set_to_none=True)

            running = 0.0
            seen = 0

            for step,batch in enumerate(
                tqdm(
                    tl,
                    desc=(
                        f"{expert} f{fold} "
                        f"e{epoch}/{EPOCHS}"
                    )
                )
            ):
                y = batch.pop("labels").to(
                    "cuda",non_blocking=True
                )
                conf = batch.pop("confidence").to(
                    "cuda",non_blocking=True
                )
                sw = batch.pop("sample_weight").to(
                    "cuda",non_blocking=True
                )
                batch.pop("source_index")

                inputs = {
                    k:v.to(
                        "cuda",non_blocking=True
                    )
                    for k,v in batch.items()
                }

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    logits = model(**inputs)

                loss = competition_shaped_loss(
                    logits,
                    y,
                    conf,
                    sw,
                    pos_weight
                )

                if not torch.isfinite(loss):
                    raise FloatingPointError(
                        "Non-finite loss: "
                        f"{expert}, fold={fold}, "
                        f"epoch={epoch}"
                    )

                scaler.scale(
                    loss/ACCUM_STEPS
                ).backward()

                running += (
                    float(loss.item())*len(y)
                )
                seen += len(y)

                if (
                    (step+1)%ACCUM_STEPS==0
                    or step+1==len(tl)
                ):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        model.parameters(),1.0
                    )

                    old_scale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()

                    if scaler.get_scale() >= old_scale:
                        scheduler.step()

                    optimizer.zero_grad(
                        set_to_none=True
                    )

            rec = {
                "expert":expert,
                "fold":fold,
                "epoch":epoch,
                "loss":running/max(seen,1),
            }
            fold_hist.append(rec)
            histories.append(rec)
            pd.DataFrame(fold_hist).to_csv(
                hist_path,index=False
            )
            print(rec)

            if epoch in SNAPSHOT_EPOCHS:
                i1,p1 = predict_probs(model,vl)
                i2,p2 = predict_probs(
                    model,val_ext_loader
                )
                i3,p3 = predict_probs(
                    model,test_ext_loader
                )

                snapshot_oof.append((i1,p1))
                snapshot_valid.append((i2,p2))
                snapshot_test.append((i3,p3))

        va_idx = snapshot_oof[0][0]
        va_prob = np.mean(
            [x[1] for x in snapshot_oof],
            axis=0
        )

        valid_idx = snapshot_valid[0][0]
        valid_prob = np.mean(
            [x[1] for x in snapshot_valid],
            axis=0
        )

        test_idx = snapshot_test[0][0]
        test_prob = np.mean(
            [x[1] for x in snapshot_test],
            axis=0
        )

        oof[va_idx] = va_prob

        valid_full = np.full(
            (len(valid),len(LABELS)),
            np.nan,dtype=np.float32
        )
        valid_full[valid_idx] = valid_prob

        test_full = np.full(
            (len(test),len(LABELS)),
            np.nan,dtype=np.float32
        )
        test_full[test_idx] = test_prob

        valid_fold_preds.append(valid_full)
        test_fold_preds.append(test_full)

        np.savez_compressed(
            cache,
            va_idx=va_idx,
            va_prob=va_prob,
            valid_idx=valid_idx,
            valid_prob=valid_prob,
            test_idx=test_idx,
            test_prob=test_prob,
        )

        del model,optimizer,scheduler,scaler
        del tl,vl,val_ext_loader,test_ext_loader
        gc.collect()
        torch.cuda.empty_cache()

    pd.DataFrame(histories).to_csv(
        RUN_DIR/f"{expert}_training_history.csv",
        index=False
    )
    pd.DataFrame(vocab_reports).to_csv(
        RUN_DIR/f"{expert}_vocab_additions.csv",
        index=False
    )

    valid_stack = np.stack(valid_fold_preds)
    test_stack = np.stack(test_fold_preds)

    if expert == "hate":
        with np.errstate(invalid="ignore"):
            valid_mean = np.nanmean(
                valid_stack,axis=0
            )
            test_mean = np.nanmean(
                test_stack,axis=0
            )
    else:
        valid_mean = valid_stack.mean(axis=0)
        test_mean = test_stack.mean(axis=0)

    np.savez_compressed(
        RUN_DIR/f"{expert}_oof_valid_test.npz",
        oof=oof,
        valid=valid_mean,
        test=test_mean
    )

    return oof,valid_mean,test_mean


In [ ]:

# 10. Train all three experts
# This is the long cell. Fold caches let you resume safely if Colab disconnects.
expert_outputs = {}

for expert in ["bangla","banglish","hate"]:
    print("\n\n"+"#"*80)
    print("STARTING EXPERT:",expert.upper())
    print("#"*80)

    expert_outputs[expert] = train_expert(expert)

print("✅ All experts completed.")


In [ ]:

# 11. Expand unique-text OOF predictions back to original train rows
bangla_oof, bangla_valid, bangla_test = expert_outputs["bangla"]
banglish_oof, banglish_valid, banglish_test = expert_outputs["banglish"]
hate_oof, hate_valid, hate_test = expert_outputs["hate"]

orig_gid = train["group_id"].to_numpy(int)
orig_folds = train["fold"].to_numpy(int)
orig_script = train["script"].to_numpy(str)
orig_y = train[LABELS].to_numpy(np.int8)

bangla_orig = bangla_oof[orig_gid]
banglish_orig = banglish_oof[orig_gid]
hate_orig = hate_oof[orig_gid]

assert np.isfinite(
    hate_orig[orig_script=="roman"]
).all()

print(
    "Original-row OOF shapes:",
    bangla_orig.shape,
    banglish_orig.shape,
    hate_orig.shape
)


In [ ]:

# 12. Fast competition metrics + cross-fitted thresholds
def f1_positive(y,pred):
    y=np.asarray(y,np.int8)
    pred=np.asarray(pred,np.int8)

    tp=np.sum((y==1)&(pred==1))
    fp=np.sum((y==0)&(pred==1))
    fn=np.sum((y==1)&(pred==0))

    den=2*tp+fp+fn
    return float(2*tp/den) if den else 0.0

def f1_macro_binary(y,pred):
    y=np.asarray(y,np.int8)
    pred=np.asarray(pred,np.int8)

    vals=[]
    for cls in [0,1]:
        tp=np.sum((y==cls)&(pred==cls))
        fp=np.sum((y!=cls)&(pred==cls))
        fn=np.sum((y==cls)&(pred!=cls))
        den=2*tp+fp+fn
        vals.append(
            2*tp/den if den else 0.0
        )

    return float(np.mean(vals))

def label_metric(label,y,pred):
    if label=="is_toxic":
        return f1_macro_binary(y,pred)
    return f1_positive(y,pred)

def best_threshold(label,y,p):
    y=np.asarray(y,np.int8)
    p=np.asarray(p,float)

    vals=np.unique(p)
    if len(vals)>700:
        vals=np.quantile(
            vals,np.linspace(0,1,700)
        )

    cand=np.unique(
        np.r_[0.0,vals,1.000001]
    )

    order=np.argsort(p)
    ps=p[order]
    ys=y[order]

    idx=np.searchsorted(
        ps,cand,side="left"
    )
    prefix_pos=np.r_[0,np.cumsum(ys)]
    prefix_neg=np.r_[0,np.cumsum(1-ys)]

    total_pos=prefix_pos[-1]
    total_neg=prefix_neg[-1]

    tp=total_pos-prefix_pos[idx]
    fp=total_neg-prefix_neg[idx]
    fn=prefix_pos[idx]
    tn=prefix_neg[idx]

    if label=="is_toxic":
        f1p=np.divide(
            2*tp,2*tp+fp+fn,
            out=np.zeros_like(tp,dtype=float),
            where=(2*tp+fp+fn)!=0,
        )
        f1n=np.divide(
            2*tn,2*tn+fp+fn,
            out=np.zeros_like(tn,dtype=float),
            where=(2*tn+fp+fn)!=0,
        )
        scores=(f1p+f1n)/2
    else:
        scores=np.divide(
            2*tp,2*tp+fp+fn,
            out=np.zeros_like(tp,dtype=float),
            where=(2*tp+fp+fn)!=0,
        )

    best=scores.max()
    tied=cand[
        np.isclose(
            scores,best,
            atol=1e-12,rtol=0
        )
    ]
    return float(np.median(tied))

def crossfit_threshold(label,y,p):
    pred=np.zeros(
        len(y),dtype=np.int8
    )
    fold_scores=[]
    thresholds=[]

    for f in range(N_FOLDS):
        held=orig_folds==f

        t=best_threshold(
            label,
            y[~held],
            p[~held]
        )
        thresholds.append(t)

        pred[held]=(
            p[held]>=t
        ).astype(np.int8)

        fold_scores.append(
            label_metric(
                label,
                y[held],
                pred[held]
            )
        )

    return {
        "score":label_metric(label,y,pred),
        "fold_scores":np.asarray(
            fold_scores,float
        ),
        "thresholds":np.asarray(
            thresholds,float
        ),
        "prediction":pred,
    }

print("Metric helpers ready.")


In [ ]:

# 13. TRAIN-ONLY script-gated expert selection
script_native = orig_script=="native"
script_mixed = orig_script=="mixed"
script_roman = orig_script=="roman"

NATIVE_BANGLA_WEIGHTS = [0.50,0.75,1.00]
MIXED_BANGLA_WEIGHTS = [0.00,0.25,0.50,0.75,1.00]
ROMAN_HATE_WEIGHTS = [0.00,0.25,0.50,0.75,1.00]

selection_rows=[]
best_by_label={}

def make_train_mix(
    label_idx,
    w_native,
    w_mixed,
    w_hate
):
    p=np.empty(len(train),dtype=float)

    p[script_native] = (
        w_native
        * bangla_orig[script_native,label_idx]
        + (1-w_native)
        * banglish_orig[script_native,label_idx]
    )

    p[script_mixed] = (
        w_mixed
        * bangla_orig[script_mixed,label_idx]
        + (1-w_mixed)
        * banglish_orig[script_mixed,label_idx]
    )

    p[script_roman] = (
        w_hate
        * hate_orig[script_roman,label_idx]
        + (1-w_hate)
        * banglish_orig[script_roman,label_idx]
    )

    return p

for j,label in enumerate(LABELS):
    y=orig_y[:,j]
    best=None

    roman_grid = (
        [0.0,0.25]
        if label=="implicit_toxicity"
        else ROMAN_HATE_WEIGHTS
    )

    for wn in NATIVE_BANGLA_WEIGHTS:
        for wm in MIXED_BANGLA_WEIGHTS:
            for wh in roman_grid:
                p=make_train_mix(
                    j,wn,wm,wh
                )
                cv=crossfit_threshold(
                    label,y,p
                )

                mean_fold=float(
                    cv["fold_scores"].mean()
                )
                std_fold=float(
                    cv["fold_scores"].std()
                )

                robust=(
                    mean_fold
                    - 0.03*std_fold
                )

                rec={
                    "label":label,
                    "w_native_bangla":wn,
                    "w_mixed_bangla":wm,
                    "w_roman_hate":wh,
                    "crossfit_score":float(
                        cv["score"]
                    ),
                    "mean_fold_score":mean_fold,
                    "std_fold_score":std_fold,
                    "robust_score":robust,
                    "fold_scores":"|".join(
                        f"{x:.6f}"
                        for x in cv["fold_scores"]
                    ),
                    "median_threshold":float(
                        np.median(
                            cv["thresholds"]
                        )
                    ),
                    "pred_positive_rate":float(
                        cv["prediction"].mean()
                    ),
                }
                selection_rows.append(rec)

                complexity=(
                    abs(wn-0.75)
                    + abs(wm-0.50)
                    + abs(wh-0.25)
                )

                key=(
                    robust,
                    cv["score"],
                    -std_fold,
                    -complexity,
                )

                if (
                    best is None
                    or key>best["key"]
                ):
                    best={
                        "key":key,
                        "rec":rec,
                        "cv":cv,
                        "train_prob":p,
                    }

    best_by_label[label]=best

selection=pd.DataFrame(
    selection_rows
).sort_values(
    ["label","robust_score"],
    ascending=[True,False]
)
selection.to_csv(
    RUN_DIR/
    "v26_train_only_ensemble_search.csv",
    index=False
)

locked=pd.DataFrame([
    b["rec"]
    for b in best_by_label.values()
])
locked.to_csv(
    RUN_DIR/"v26_locked_config.csv",
    index=False
)

print(
    "LOCKED CONFIG — selected WITHOUT "
    "validation labels"
)
display(locked.round(5))


In [ ]:

# 14. Build locked validation/test probabilities and predictions
def make_external_mix(
    label_idx,
    script_values,
    bangla_p,
    banglish_p,
    hate_p,
    rec
):
    out=np.empty(
        len(script_values),
        dtype=float
    )

    native=script_values=="native"
    mixed=script_values=="mixed"
    roman=script_values=="roman"

    wn=float(
        rec["w_native_bangla"]
    )
    wm=float(
        rec["w_mixed_bangla"]
    )
    wh=float(
        rec["w_roman_hate"]
    )

    out[native] = (
        wn*bangla_p[native,label_idx]
        + (1-wn)
        * banglish_p[native,label_idx]
    )

    out[mixed] = (
        wm*bangla_p[mixed,label_idx]
        + (1-wm)
        * banglish_p[mixed,label_idx]
    )

    out[roman] = (
        wh*hate_p[roman,label_idx]
        + (1-wh)
        * banglish_p[roman,label_idx]
    )

    return out

valid_script=valid["script"].to_numpy(str)
test_script=test["script"].to_numpy(str)

valid_probs=np.zeros(
    (len(valid),len(LABELS)),float
)
test_probs=np.zeros(
    (len(test),len(LABELS)),float
)

valid_pred=np.zeros(
    (len(valid),len(LABELS)),np.int8
)
test_pred=np.zeros(
    (len(test),len(LABELS)),np.int8
)

for j,label in enumerate(LABELS):
    rec=best_by_label[label]["rec"]

    valid_probs[:,j]=make_external_mix(
        j,
        valid_script,
        bangla_valid,
        banglish_valid,
        hate_valid,
        rec,
    )

    test_probs[:,j]=make_external_mix(
        j,
        test_script,
        bangla_test,
        banglish_test,
        hate_test,
        rec,
    )

    threshold=float(
        rec["median_threshold"]
    )

    valid_pred[:,j]=(
        valid_probs[:,j]>=threshold
    ).astype(np.int8)

    test_pred[:,j]=(
        test_probs[:,j]>=threshold
    ).astype(np.int8)

print("Locked predictions created.")


In [ ]:

# 15. ONE-SHOT validation evaluation — NO tuning after this point
y_valid=valid[LABELS].to_numpy(
    np.int8
)

per_label=[]
for j,label in enumerate(LABELS):
    s=label_metric(
        label,
        y_valid[:,j],
        valid_pred[:,j]
    )
    per_label.append(s)

tox_score=per_label[0]
attr_score=float(
    np.mean(per_label[1:])
)
official=(
    0.5*tox_score
    + 0.5*attr_score
)

valid_report=pd.DataFrame({
    "label":LABELS,
    "validation_score":per_label,
    "true_positive_rate":y_valid.mean(
        axis=0
    ),
    "pred_positive_rate":valid_pred.mean(
        axis=0
    ),
})
valid_report.to_csv(
    RUN_DIR/"v26_one_shot_validation.csv",
    index=False
)

summary={
    "toxicity_macro_f1":float(
        tox_score
    ),
    "attribute_mean_f1":float(
        attr_score
    ),
    "official_validation_score":float(
        official
    ),
    "IMPORTANT":(
        "Validation was untouched during "
        "model/weight/threshold selection."
    )
}

(RUN_DIR/"v26_validation_summary.json").write_text(
    json.dumps(summary,indent=2)
)

print(json.dumps(summary,indent=2))
display(valid_report.round(5))


In [ ]:

# 16. Script-specific one-shot validation diagnostics
rows=[]

for j,label in enumerate(LABELS):
    for script in [
        "native","mixed","roman"
    ]:
        mask=valid_script==script
        if mask.sum()==0:
            continue

        rows.append({
            "label":label,
            "script":script,
            "rows":int(mask.sum()),
            "positives":int(
                y_valid[mask,j].sum()
            ),
            "score":label_metric(
                label,
                y_valid[mask,j],
                valid_pred[mask,j]
            ),
            "pred_positive_rate":float(
                valid_pred[mask,j].mean()
            ),
        })

script_report=pd.DataFrame(rows)
script_report.to_csv(
    RUN_DIR/
    "v26_validation_by_script.csv",
    index=False
)
display(script_report.round(5))


In [ ]:

# 17. Build V26 submission + diagnostics
submission=champion.copy()

for j,label in enumerate(LABELS):
    submission[label]=(
        test_pred[:,j].astype(int)
    )

assert (
    submission["ID"].tolist()
    == test["ID"].tolist()
)
assert (
    submission[LABELS]
    .isin([0,1])
    .all()
    .all()
)

submission.to_csv(
    RUN_DIR/"submission_v26_locked.csv",
    index=False
)

comparison=[]
for label in LABELS:
    old=champion[label].to_numpy(int)
    new=submission[label].to_numpy(int)

    comparison.append({
        "label":label,
        "v14_positives":int(
            old.sum()
        ),
        "v26_positives":int(
            new.sum()
        ),
        "changed_rows":int(
            (old!=new).sum()
        ),
        "v14_positive_rate":float(
            old.mean()
        ),
        "v26_positive_rate":float(
            new.mean()
        ),
    })

comparison=pd.DataFrame(comparison)
comparison.to_csv(
    RUN_DIR/
    "v26_vs_v14_test_diagnostics.csv",
    index=False
)

print("V26 vs V14 test diagnostics")
display(comparison)

np.savez_compressed(
    RUN_DIR/"v26_probabilities.npz",
    validation=valid_probs,
    test=test_probs
)


In [ ]:

# 18. Final report and package
final_report = {
    "version":"v26_noise_aware_dual_script_moe",
    "protected_public_lb_champion":0.586,
    "models":MODELS,
    "selection_data":"train only",
    "validation_usage":(
        "one-shot evaluation only; "
        "never used for tuning"
    ),
    "external_labeled_data_used":False,
    "deduplicated_soft_label_training":True,
    "fold_local_dataset_specific_tokens":True,
    "official_validation_score":float(
        official
    ),
    "toxicity_macro_f1":float(
        tox_score
    ),
    "attribute_mean_f1":float(
        attr_score
    ),
    "submission_file":"submission_v26_locked.csv",
    "instruction":(
        "Send v26_results.zip to ChatGPT "
        "before Kaggle submission."
    )
}

(RUN_DIR/"v26_report.json").write_text(
    json.dumps(final_report,indent=2)
)

zip_path=Path(
    "/content/v26_results.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(
                p,
                arcname=p.name
            )

print(json.dumps(final_report,indent=2))
print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
